**Loading Llama3.2 model**

In [ ]:
!pip install huggingface_hub
!mkdir -p models

from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id="hugging-quants/Llama-3.2-1B-Instruct-Q8_0-GGUF",
    filename="llama-3.2-1b-instruct-q8_0.gguf",
    local_dir="models"
)

llama-3.2-1b-instruct-q8_0.gguf: reconstructing file:   0%|          |  0.00B / 1.32GB            

llama-3.2-1b-instruct-q8_0.gguf: downloading bytes:           |  0.00B            

'/content/models/llama-3.2-1b-instruct-q8_0.gguf'

In [ ]:
!pip install llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

from llama_cpp import Llama
model = Llama(
    model_path="models/llama-3.2-1b-instruct-q8_0.gguf",
    n_gpu_layers=-1,
    n_ctx=2048,
    seed=42,
    verbose=False,
)

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu121


Generating response for the prompt

In [8]:
def generate(messages, max_new_tokens=200, do_sample=True, temperature=1, top_p=0.25):
    response = model.create_chat_completion(
        messages=messages,
        max_tokens=max_new_tokens,
        temperature=temperature if do_sample else 0.0,
        top_p=top_p,
    )
    return response["choices"][0]["message"]["content"]

In [9]:
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]
print(generate(messages))

Here's one:

Why did the chicken go to the doctor?

Because it had a fowl cough!

(Sorry, I know it's a bit of a "egg-xistential" crisis, but I hope it cracked you up!)


In [10]:
messages = [{"role": "user", "content": "Classify the text into neutral, negative or positive.\nText: I think the food was okay.\nSentiment:"}]
print(generate(messages))

The text is neutral. The word "okay" is a neutral adverb that doesn't convey any strong emotions or opinions. It simply indicates that the food was satisfactory, without expressing any positive or negative feelings.


In [11]:
persona = "You are an expert in AI programming assistant.Help solving, writing, explaining any code to make the user's work easy.\n"
instruction= "Give a step by step answer for the user's question.If it's a coding, answer should be working code.\n"
context = "The assistance is used by the some developers.\n"
data_format = """1. Question summery.
2. Explaination
3. Code (if applicabel)
4. Conclusion\n"""
audience = "The user is beginner to intermediate programer.\n"
tone = "Friendly, professional, and concise.\n"
data = "Explain me a C program for sum of two numbers in a simple way with code."
full_prompt = persona + instruction + context + data_format + audience + tone + data
messages = [{"role": "user", "content": full_prompt}]
print(generate(messages,max_new_tokens=350))

I'd be happy to help you with a simple C program that calculates the sum of two numbers.

**Question Summary:**

* Write a C program that takes two numbers as input from the user.
* Calculates the sum of these two numbers.
* Outputs the result to the console.

**Explanation:**

This program uses basic C syntax and data types. We'll use two variables to store the input numbers and a conditional statement to calculate the sum.

**Code:**
```c
#include <stdio.h>

int main() {
    int num1, num2;

    // Prompt the user to enter two numbers
    printf("Enter the first number: ");
    scanf("%d", &num1);

    printf("Enter the second number: ");
    scanf("%d", &num2);

    // Calculate the sum
    int sum = num1 + num2;

    // Output the result
    printf("The sum of %d and %d is: %d\n", num1, num2, sum);

    return 0;
}
```
**Explanation:**

1. We include the `stdio.h` header file to use the `printf` and `scanf` functions.
2. We declare two integer variables `num1` and `num2` to store t

**Chain-of-Thought — zero-shot version**

In [12]:
zeroshot_cot_prompt = [
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."}
]
print(generate(zeroshot_cot_prompt, max_new_tokens=200))

Let's think step-by-step to solve this problem.

Step 1: Start with the number of apples the cafeteria had initially.
The cafeteria had 23 apples.

Step 2: Subtract the number of apples used to make lunch.
They used 20 apples to make lunch, so they have 23 - 20 = 3 apples left.

Step 3: Add the number of apples bought.
They bought 6 more apples, so they now have 3 + 6 = 9 apples.

The final answer is 9.


**Tree-of-Thought**

In [15]:
zeroshot_tot_prompt = [
    {"role": "user", "content": (
        "Imagine three different experts are answering this question. "
        "All experts will write down 1 step of their thinking, then share it with the group. "
        "Then all experts will go on to the next step, etc. "
        "If any expert realizes they're wrong at any point then they leave. "
        "The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, "
        "how many apples do they have?' Make sure to discuss the results in short."
    )}
]
print(generate(zeroshot_tot_prompt, max_new_tokens=500))

Here are the three experts' steps:

**Expert 1: John**
Step 1: The cafeteria started with 23 apples.
Step 2: They used 20 apples to make lunch, leaving 3 apples.
Step 3: They bought 6 more apples, bringing the total to 3 + 6 = 9 apples.

**Expert 2: Sarah**
Step 1: The cafeteria started with 23 apples.
Step 2: They used 20 apples to make lunch, leaving 3 apples.
Step 3: They bought 6 more apples, bringing the total to 3 + 6 = 9 apples.

**Expert 3: Michael**
Step 1: The cafeteria started with 23 apples.
Step 2: They used 20 apples to make lunch, leaving 3 apples.
Step 3: They bought 6 more apples, bringing the total to 3 + 6 = 9 apples.

**Expert 1: John**
Step 1: The cafeteria started with 23 apples.
Step 2: They used 20 apples to make lunch, leaving 3 apples.
Step 3: They bought 6 more apples, bringing the total to 3 + 6 = 9 apples.

**Expert 2: Sarah**
Step 1: The cafeteria started with 23 apples.
Step 2: They used 20 apples to make lunch, leaving 3 apples.
Step 3: They bought 6 mor

**Output validation**

In [14]:
one_shot_template = """Create a short character profile for an RPG game. Make sure to only use this format:
{
  "description": "A SHORT DESCRIPTION",
  "name": "THE CHARACTER'S NAME",
  "armor": "ONE PIECE OF ARMOR",
  "weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [{"role": "user", "content": one_shot_template}]
output = generate(one_shot_prompt, max_new_tokens=150)
print(output)

{
  "description": "A skilled warrior with a mysterious past.",
  "name": "Kaito Yamato",
  "armor": "ONE PIECE OF ARMOR",
  "weapon": "SWORD"
}
